In [ ]:
#imports
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
    KFold,
    cross_val_score
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from lightgbm import LGBMRegressor

import matplotlib.pyplot as plt

import joblib

In [ ]:
from pathlib import Path
import os

def _find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "README.md").exists() and (candidate / "notebooks").is_dir():
            return candidate
    return current

PROJECT_ROOT = _find_project_root()
DATA_ROOT = Path(os.environ.get("GANGA_DATA_ROOT", PROJECT_ROOT / "data")).expanduser()
OUTPUT_ROOT = Path(os.environ.get("GANGA_OUTPUT_ROOT", PROJECT_ROOT / "outputs")).expanduser()

PROJECT_DIR = PROJECT_ROOT
ML_READY_DATASET = OUTPUT_ROOT / "cwc" / "ML_Ready_Dataset.csv"
df = pd.read_csv(ML_READY_DATASET)
print(df.shape)
df.head()


In [ ]:
import re

X.columns = [
    re.sub(r"[^A-Za-z0-9_]", "_", str(c))
    for c in X.columns
]

print(X.columns[:15])

In [ ]:
#split features & target
TARGET = "Total Coliform (MPN/100mL)"

X = df.drop(columns=[TARGET])

y = df[TARGET]

print(X.shape)
print(y.shape)

In [ ]:
#train test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

In [ ]:
# #Hyperparameter Search
# params = {

#     "n_estimators":[200,400,600,800],

#     "learning_rate":[
#         0.01,
#         0.03,
#         0.05,
#         0.1
#     ],

#     "num_leaves":[
#         20,
#         31,
#         40,
#         60
#     ],

#     "max_depth":[
#         -1,
#         6,
#         8,
#         10,
#         12
#     ],

#     "subsample":[
#         0.7,
#         0.8,
#         0.9,
#         1.0
#     ],

#     "colsample_bytree":[
#         0.7,
#         0.8,
#         0.9,
#         1.0
#     ]
# }

In [ ]:
#Randomized Search
search = RandomizedSearchCV(

    estimator=LGBMRegressor(random_state=42),

    param_distributions=params,

    n_iter=30,

    scoring="r2",

    cv=5,

    random_state=42,

    verbose=2,

    n_jobs=-1
)

In [ ]:
#train
search.fit(
    X_train.values,
    y_train.values
)

In [ ]:
print(search.best_score_)

print(search.best_params_)

In [ ]:
best_model = search.best_estimator_

pred = best_model.predict(
    X_test.values
)

mae = mean_absolute_error(
    y_test,
    pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        pred
    )
)

r2 = r2_score(
    y_test,
    pred
)

print("MAE :",mae)
print("RMSE:",rmse)
print("R2  :",r2)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,7))

plt.scatter(
    y_test,
    pred,
    alpha=0.5
)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="red"
)

plt.xlabel("Actual Total Coliform")
plt.ylabel("Predicted Total Coliform")
plt.title("Actual vs Predicted")

plt.show()

In [ ]:
residual = y_test - pred

plt.figure(figsize=(8,5))

plt.scatter(
    pred,
    residual,
    alpha=0.5
)

plt.axhline(
    y=0,
    color="red"
)

plt.xlabel("Predicted")
plt.ylabel("Residual")
plt.title("Residual Plot")

plt.show()

In [ ]:
import joblib

joblib.dump(
    best_model,
    "Outputs/LightGBM_Final.joblib"
)

In [ ]:
#save best parameters
joblib.dump(
    search.best_params_,
    "Outputs/LightGBM_Best_Params.pkl"
)

In [ ]:
print("="*60)
print("FINAL MODEL")
print("="*60)

print("Best Parameters")
print(search.best_params_)

print()

print("MAE :", mae)
print("RMSE:", rmse)
print("R2  :", r2)

print("="*60)

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
X_train
X_test
y_train
y_test

In [ ]:
param_grid = {
    "n_estimators": [200,300,500,800],
    "learning_rate": [0.03,0.05,0.08,0.1],
    "num_leaves": [20,31,50,70,100],
    "max_depth": [-1,5,8,10,12],
    "min_child_samples": [10,20,40,80],
    "subsample": [0.7,0.8,0.9,1.0],
    "colsample_bytree": [0.6,0.7,0.8,1.0],
    "reg_alpha":[0,0.1,0.5,1],
    "reg_lambda":[0,0.1,0.5,1]
}

search = RandomizedSearchCV(
    estimator=LGBMRegressor(random_state=42),
    param_distributions=param_grid,
    n_iter=40,
    cv=5,
    scoring="r2",
    random_state=42,
    n_jobs=-1,
    verbose=2
)

search.fit(
    X_train.values,
    y_train.values
)

In [ ]:
print(search.best_score_)
print(search.best_params_)

In [ ]:
best_model = search.best_estimator_

pred = best_model.predict(X_test.values)

from sklearn.metrics import *

print("MAE :", mean_absolute_error(y_test,pred))
print("RMSE:", mean_squared_error(y_test,pred)**0.5)
print("R2 :", r2_score(y_test,pred))

In [ ]:
import joblib

joblib.dump(
    best_model,
    "Outputs/LightGBM_Final.joblib"
)

print("Model Saved Successfully")

In [ ]:
joblib.dump(
    search.best_params_,
    "Outputs/LightGBM_Best_Params.pkl"
)

print("Parameters Saved Successfully")

In [ ]:
importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": best_model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

display(importance.head(20))

In [ ]:
import matplotlib.pyplot as plt

top = importance.head(20)

plt.figure(figsize=(10,8))

plt.barh(
    top["Feature"],
    top["Importance"]
)

plt.gca().invert_yaxis()

plt.title("Top 20 Important Features")

plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(7,7))

plt.scatter(
    y_test,
    pred,
    alpha=0.5
)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="red"
)

plt.xlabel("Actual")
plt.ylabel("Predicted")

plt.title("Actual vs Predicted")

plt.show()

In [ ]:
residual = y_test - pred

plt.figure(figsize=(8,5))

plt.scatter(
    pred,
    residual,
    alpha=0.5
)

plt.axhline(
    0,
    color="red"
)

plt.xlabel("Predicted")
plt.ylabel("Residual")

plt.title("Residual Plot")

plt.show()